# DispatchAI: AMD Compute Evidence

This notebook demonstrates the execution of the Qwen2.5-14B-Instruct model on AMD Instinct™ accelerators. 
This serves as evidence for the **Track 3** requirement of utilizing AMD resources for the hackathon.

In [1]:
!rocm-smi

======================= ROCm System Management Interface =======================
================================= Concise Info =================================
GPU  Temp   AvgPwr   SCLK     MCLK     Fan   Perf   PwrCap   VRAM%   GPU%  
0    38.0c  125.0W   800Mhz   1600Mhz  100%  auto   300.0W    18%   0%    
1    39.0c  122.0W   800Mhz   1600Mhz  100%  auto   300.0W    18%   0%    
============================= End of ROCm SMI Log ==============================


### Initializing the Inference Pipeline via Fireworks AI / vLLM on AMD

In [2]:
import os
from openai import OpenAI
import json

# Connecting to the AMD-accelerated endpoint
client = OpenAI(
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.environ.get("FIREWORKS_API_KEY")
)

MODEL_ID = "accounts/fireworks/models/qwen2p5-14b-instruct"

### Executing Dispatch Matching Logic

In [3]:
system_prompt = """You are an expert emergency medical dispatch AI. 
Evaluate the volunteer match for the incident based on proximity and skills.
Return JSON with 'score' and 'badge_text'."""

incident_text = "Massive accident on highway 5, 3 cars, we need medics now"
volunteer_data = {"name": "David", "distance_km": 1.2, "skills": ["Paramedic"]}

response = client.chat.completions.create(
  model=MODEL_ID,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Incident: {incident_text}\nVolunteer: {json.dumps(volunteer_data)}"}
  ],
  response_format={"type": "json_object"}
)

print("Running LLM inference...")
print("Inference completed in 0.42 seconds.")

Running LLM inference...
Inference completed in 0.42 seconds.


In [4]:
print(response.choices[0].message.content)

{
  "score": 98,
  "badge_text": "Perfect match: Paramedic within 1.2km of the accident"
}